In [ ]:
# Install required packages (run once)
%pip install numpy pandas scikit-learn xgboost lightgbm catboost plotly -q

# LTC Expected Loss - Comparative Model Analysis

Predicting **Expected Loss** per policyholder:

$$\text{Expected Loss} = P(\text{Claim}) \times E[\text{Payout}\mid\text{Claim}]$$

We compare a few modeling strategies here:

| Strategy | Models |
|---|---|
| **Two-Part (Freq x Sev)** | GLM, XGBoost, LightGBM, CatBoost, Random Forest |
| **Single Model (Tweedie)** | GLM, XGBoost, LightGBM (all with Tweedie objective) |

All tree-based models go through **hyperparameter grid search** (3-fold CV). Evaluation: Gini, calibration ratio, decile lift.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings; warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

# Import all model functions from our module
from ltc_models import (
 load_and_prepare_data,
 train_frequency_glm, train_frequency_xgb,
 train_frequency_lgbm, train_frequency_catboost, train_frequency_rf,
 train_severity_xgb, train_severity_lgbm,
 train_severity_catboost, train_severity_rf,
 train_tweedie, train_tweedie_xgb, train_tweedie_lgbm,
 evaluate_frequency, evaluate_severity, evaluate_pure_premium,
 normalized_gini, compare_models, get_frequency_proba,
)

# Load and prepare data
df, X, FEATURES, idx_tr, idx_te = load_and_prepare_data()
print(f"Dataset: {df.shape[0]:,} policies | Train: {len(idx_tr):,} | Test: {len(idx_te):,}")
print(f"Features ({len(FEATURES)}): {FEATURES}")

Dataset: 50,000 policies | Train: 37,500 | Test: 12,500
Features (13): ['Customer_Age', 'Max_Daily_Benefit_USD', 'Risk_Score_Tier', 'Caregiver_Availability_Index', 'Macro_Inflation_Rate', 'Prior_Claims_Count', 'Age_x_Risk', 'High_Risk_Flag', 'Claims_Per_Year', 'Log_Benefit', 'Low_Caregiver_x_Age', 'Setting_Home Care', 'Setting_Nursing Home']


## 1. Exploratory Data Analysis

In [2]:
# Data quality & key statistics
print("=== Data Quality ===")
print(f"Missing values: {df.isna().sum().sum()}")
print(f"Claim=1 but payout=0: {((df.Claim_Occurred==1)&(df.Total_LTC_Payout_USD==0)).sum()}")
print(f"Claim=0 but payout>0: {((df.Claim_Occurred==0)&(df.Total_LTC_Payout_USD>0)).sum()}")

print("\n=== Key Statistics ===")
claim_rate = df.Claim_Occurred.mean()
print(f"Overall claim frequency : {claim_rate:.2%}")
print(f"Share of $0 payouts  : {(df.Total_LTC_Payout_USD==0).mean():.2%}")
print(f"\nSeverity among claimants (USD):")
print(df.loc[df.Claim_Occurred==1, "Total_LTC_Payout_USD"].describe())

=== Data Quality ===
Missing values: 0
Claim=1 but payout=0: 0
Claim=0 but payout>0: 0

=== Key Statistics ===
Overall claim frequency : 13.94%
Share of $0 payouts  : 86.06%

Severity among claimants (USD):
count       6,972.0000
mean      288,869.1206
std       226,714.9225
min         9,289.6600
25%       134,990.9775
50%       230,337.5200
75%       371,899.3650
max     3,185,324.0500
Name: Total_LTC_Payout_USD, dtype: float64


In [3]:
# Visualize the zero-inflation and severity distribution
sev = df.loc[df.Claim_Occurred == 1, "Total_LTC_Payout_USD"]

fig = make_subplots(rows=1, cols=2, subplot_titles=[
 "Frequency: 86% of policies never claim",
 f"Severity | claim - right-skewed (mean ${sev.mean():,.0f})"
])

counts = df.Claim_Occurred.value_counts().sort_index()
fig.add_trace(go.Bar(x=["No Claim", "Claim"], y=counts.values,
      marker_color=["#9bbcd1", "#1f6f8b"]), row=1, col=1)
fig.add_trace(go.Histogram(x=sev, nbinsx=60, marker_color="#1f6f8b",
       name="Payout"), row=1, col=2)
fig.update_layout(height=400, showlegend=False,
     template="plotly_white", title_text="Claim Distribution")
fig.update_xaxes(title_text="Payout (USD)", row=1, col=2)
fig.show()

In [4]:
# Risk drivers: claim frequency by key segments
for col in ["Risk_Score_Tier", "Care_Setting_Preference","Prior_Claims_Count"]:
 print(f"\nClaim frequency by {col}:")
 print(df.groupby(col).Claim_Occurred.mean().sort_values(ascending=False).map(lambda v: f"{v:.2%}"))

df["_AgeBand"] = pd.cut(df.Customer_Age, [39, 50, 60, 70, 80, 86])
print("\nClaim frequency by age band:")
print(df.groupby("_AgeBand").Claim_Occurred.mean().map(lambda v: f"{v:.2%}"))
df.drop(columns="_AgeBand", inplace=True)


Claim frequency by Risk_Score_Tier:
Risk_Score_Tier
5    23.47%
4    19.19%
3    14.87%
2    12.36%
1     9.18%
Name: Claim_Occurred, dtype: object

Claim frequency by Care_Setting_Preference:
Care_Setting_Preference
Nursing Home       14.51%
Home Care          14.08%
Assisted Living    13.50%
Name: Claim_Occurred, dtype: object

Claim frequency by Prior_Claims_Count:
Prior_Claims_Count
2    26.11%
1    18.85%
0    11.85%
Name: Claim_Occurred, dtype: object

Claim frequency by age band:
_AgeBand
(39, 50]     3.56%
(50, 60]     7.29%
(60, 70]    13.45%
(70, 80]    23.01%
(80, 86]    33.43%
Name: Claim_Occurred, dtype: object


**EDA notes:**
- ~14% claim rate, heavy-tailed severity (mean ~$300K, some claims >$3M)
- Age, prior claims, and risk tier are the strongest signals
- Care setting shifts both frequency and cost

---
## 2. Frequency Models - P(Claim)

Training 5 models with grid search (3-fold CV on AUC). Comparing on **Gini** (= 2*AUC - 1).

In [5]:
# Prepare frequency targets
X_train, X_test = X.loc[idx_tr], X.loc[idx_te]
yf_tr = df.loc[idx_tr, "Claim_Occurred"]
yf_te = df.loc[idx_te, "Claim_Occurred"]

print("Training 5 frequency models with grid search...")
print("=" * 60)

# 1. GLM (Logistic Regression)
print("\n[1/5] Logistic GLM...")
freq_glm, params_glm = train_frequency_glm(X_train, yf_tr)
print("  No grid search needed (baseline)")

# 2. XGBoost
print("\n[2/5] XGBoost (grid search)...")
freq_xgb, params_xgb = train_frequency_xgb(X_train, yf_tr)
print(f"  Best params: {params_xgb}")

# 3. LightGBM
print("\n[3/5] LightGBM (grid search)...")
freq_lgbm, params_lgbm = train_frequency_lgbm(X_train, yf_tr)
print(f"  Best params: {params_lgbm}")

# 4. CatBoost
print("\n[4/5] CatBoost (grid search)...")
freq_catboost, params_catboost = train_frequency_catboost(X_train, yf_tr)
print(f"  Best params: {params_catboost}")

# 5. Random Forest
print("\n[5/5] Random Forest (grid search)...")
freq_rf, params_rf = train_frequency_rf(X_train, yf_tr)
print(f"  Best params: {params_rf}")

print("\n" + "=" * 60)
print("All frequency models trained.")

Training 5 frequency models with grid search...

[1/5] Logistic GLM...
  No grid search needed (baseline)

[2/5] XGBoost (grid search)...
  Best params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200}

[3/5] LightGBM (grid search)...
  Best params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200, 'num_leaves': 15}

[4/5] CatBoost (grid search)...
  Best params: {'depth': 4, 'iterations': 200, 'learning_rate': 0.05}

[5/5] Random Forest (grid search)...
  Best params: {'max_depth': 5, 'min_samples_leaf': 10, 'n_estimators': 300}

All frequency models trained.


In [6]:
# Evaluate all frequency models
freq_models = {
 "GLM (Logistic)": freq_glm,
 "XGBoost": freq_xgb,
 "LightGBM": freq_lgbm,
 "CatBoost": freq_catboost,
 "Random Forest": freq_rf,
}

freq_results = []
freq_probas = {}
for name, model in freq_models.items():
 p = get_frequency_proba(model, X_test)
 freq_probas[name] = p
 freq_results.append(evaluate_frequency(yf_te, p, model_name=name))

freq_comparison = compare_models(freq_results)
print("=== Frequency Model Comparison ===\n")
print(freq_comparison.to_string())
print(f"\nActual test frequency: {yf_te.mean():.4f}")

=== Frequency Model Comparison ===

                  AUC   Gini  Pred_Freq  Actual_Freq
Model                                               
GLM (Logistic) 0.7714 0.5427     0.1406       0.1394
XGBoost        0.7686 0.5373     0.1399       0.1394
LightGBM       0.7690 0.5380     0.1401       0.1394
CatBoost       0.7697 0.5394     0.1402       0.1394
Random Forest  0.7670 0.5340     0.1403       0.1394

Actual test frequency: 0.1394


In [7]:
# Visual comparison of frequency Gini scores (Plotly)
freq_df = pd.DataFrame(freq_results)
fig = px.bar(freq_df, x="Model", y="Gini", text="Gini",
    color_discrete_sequence=["#1f6f8b"],
    title="Frequency Model Comparison - Gini Coefficient")
fig.update_traces(textposition="outside", texttemplate="%{text:.4f}")
fig.update_layout(template="plotly_white", height=400, yaxis_title="Gini")
fig.show()

---
## 3. Severity Models - E[Payout | Claim]

Trained only on the ~7K policies that actually claimed. Payouts are strictly positive and right-skewed, so tree models use **Gamma objective** where available. Grid search optimized on MAE.

In [8]:
# Prepare severity targets (claimants only)
clm_tr = idx_tr[df.loc[idx_tr, "Claim_Occurred"] == 1]
clm_te = idx_te[df.loc[idx_te, "Claim_Occurred"] == 1]
ys_tr = df.loc[clm_tr, "Total_LTC_Payout_USD"]
ys_te = df.loc[clm_te, "Total_LTC_Payout_USD"]
print(f"Severity train: {len(clm_tr):,} claimants | Test: {len(clm_te):,} claimants")

print("\nTraining 4 severity models with grid search...")
print("=" * 60)

# 1. XGBoost (Gamma)
print("\n[1/4] XGBoost Gamma...")
sev_xgb, sparams_xgb = train_severity_xgb(X.loc[clm_tr], ys_tr)
print(f"  Best params: {sparams_xgb}")

# 2. LightGBM (Gamma)
print("\n[2/4] LightGBM Gamma...")
sev_lgbm, sparams_lgbm = train_severity_lgbm(X.loc[clm_tr], ys_tr)
print(f"  Best params: {sparams_lgbm}")

# 3. CatBoost
print("\n[3/4] CatBoost...")
sev_catboost, sparams_catboost = train_severity_catboost(X.loc[clm_tr], ys_tr)
print(f"  Best params: {sparams_catboost}")

# 4. Random Forest
print("\n[4/4] Random Forest...")
sev_rf, sparams_rf = train_severity_rf(X.loc[clm_tr], ys_tr)
print(f"  Best params: {sparams_rf}")

print("\n" + "=" * 60)
print("All severity models trained.")

Severity train: 5,229 claimants | Test: 1,743 claimants

Training 4 severity models with grid search...

[1/4] XGBoost Gamma...
  Best params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200}

[2/4] LightGBM Gamma...
  Best params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200, 'num_leaves': 15}

[3/4] CatBoost...
  Best params: {'depth': 4, 'iterations': 200, 'learning_rate': 0.05}

[4/4] Random Forest...
  Best params: {'max_depth': 5, 'min_samples_leaf': 10, 'n_estimators': 300}

All severity models trained.


In [9]:
# Evaluate all severity models
sev_models = {
 "XGBoost (Gamma)": sev_xgb,
 "LightGBM (Gamma)": sev_lgbm,
 "CatBoost": sev_catboost,
 "Random Forest": sev_rf,
}

sev_results = []
sev_preds = {}
for name, model in sev_models.items():
 pred = model.predict(X.loc[clm_te])
 sev_preds[name] = pred
 sev_results.append(evaluate_severity(ys_te.values, pred, model_name=name))

sev_comparison = compare_models(sev_results)
print("=== Severity Model Comparison ===\n")
print(sev_comparison.to_string())

=== Severity Model Comparison ===

                         RMSE          MAE  Mean_Actual
Model                                                  
XGBoost (Gamma)  177,438.0000 120,041.0000 300,113.0000
LightGBM (Gamma) 177,425.0000 120,208.0000 300,113.0000
CatBoost         176,472.0000 120,629.0000 300,113.0000
Random Forest    178,413.0000 122,370.0000 300,113.0000


In [10]:
# Visual comparison of severity MAE (Plotly)
sev_df = pd.DataFrame(sev_results)
fig = px.bar(sev_df, x="Model", y="MAE", text="MAE",
    color_discrete_sequence=["#d4a843"],
    title="Severity Model Comparison - Mean Absolute Error")
fig.update_traces(textposition="outside", texttemplate="$%{text:,.0f}")
fig.update_layout(template="plotly_white", height=400, yaxis_title="MAE (USD)")
fig.show()

---
## 4. Tweedie Model - Direct Pure Premium

Tweedie (power between 1-2) is a compound Poisson-Gamma that handles zero-inflated continuous targets directly. Instead of splitting into freq/sev, it models E[Loss] in one shot.

We train three variants:
1. **Tweedie GLM** - Linear baseline, interpretable
2. **XGBoost Tweedie** - Gradient boosted trees with reg:tweedie objective
3. **LightGBM Tweedie** - Histogram-based boosting with Tweedie loss

This tells us whether the tree-based single model can match or beat the two-part approach.

In [11]:
# Train Tweedie models on full target (including zeros)
y_loss_tr = df.loc[idx_tr, "Total_LTC_Payout_USD"]
y_loss_te = df.loc[idx_te, "Total_LTC_Payout_USD"]

print("Training 3 Tweedie models (direct pure premium)...")
print("=" * 60)

# 1. Tweedie GLM
print("\n[1/3] Tweedie GLM (grid search over power & alpha)...")
tweedie_glm, tweedie_glm_params = train_tweedie(X_train, y_loss_tr)
print(f"  Best params: {tweedie_glm_params}")

# 2. XGBoost Tweedie
print("\n[2/3] XGBoost Tweedie (grid search)...")
tweedie_xgb_model, tweedie_xgb_params = train_tweedie_xgb(X_train, y_loss_tr)
print(f"  Best params: {tweedie_xgb_params}")

# 3. LightGBM Tweedie
print("\n[3/3] LightGBM Tweedie (grid search)...")
tweedie_lgbm_model, tweedie_lgbm_params = train_tweedie_lgbm(X_train, y_loss_tr)
print(f"  Best params: {tweedie_lgbm_params}")

print("\n" + "=" * 60)

# Evaluate all Tweedie models
tweedie_models_dict = {
 "Tweedie GLM": tweedie_glm,
 "Tweedie XGBoost": tweedie_xgb_model,
 "Tweedie LightGBM": tweedie_lgbm_model,
}

tweedie_results = []
tweedie_preds = {}
for name, model in tweedie_models_dict.items():
 pred = model.predict(X_test)
 tweedie_preds[name] = pred
 result = evaluate_pure_premium(y_loss_te.values, pred, model_name=name)
 tweedie_results.append(result)
 print(f"\n{name}:")
 print(f" Calibration Ratio : {result['Calibration_Ratio']:.4f}")
 print(f" Normalized Gini : {result['Normalized_Gini']:.4f}")

best_tweedie = max(tweedie_results, key=lambda x: x["Normalized_Gini"])
print(f"\nBest single-model Tweedie: {best_tweedie['Model']} (Gini={best_tweedie['Normalized_Gini']:.4f})")


Training 3 Tweedie models (direct pure premium)...

[1/3] Tweedie GLM (grid search over power & alpha)...
  Best params: {'alpha': 10.0, 'power': 1.5}

[2/3] XGBoost Tweedie (grid search)...
  Best params: {'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 300, 'tweedie_variance_power': 1.7}

[3/3] LightGBM Tweedie (grid search)...
  Best params: {'learning_rate': 0.05, 'n_estimators': 300, 'num_leaves': 50, 'tweedie_variance_power': 1.7}


Tweedie GLM:
 Calibration Ratio : 0.9800
 Normalized Gini : 0.6780

Tweedie XGBoost:
 Calibration Ratio : 0.8023
 Normalized Gini : 0.6461

Tweedie LightGBM:
 Calibration Ratio : 0.7439
 Normalized Gini : 0.6389

Best single-model Tweedie: Tweedie GLM (Gini=0.6780)


---
## 5. Pure Premium Comparison - All Two-Part Combinations + Tweedie

For each frequency model, we pair it with each severity model to compute: `Pure Premium = P(Claim) - E[Severity]`. We then evaluate calibration and Gini on the held-out test set.

In [12]:
# Compute pure premium for all freq x sev combinations
actual_loss = df.loc[idx_te, "Total_LTC_Payout_USD"].values
pp_results = []

# Best combination trackers
best_gini = -1
best_combo_name = ""
best_pp = None

for f_name, f_model in freq_models.items():
 p_freq = get_frequency_proba(f_model, X_test)
 for s_name, s_model in sev_models.items():
  sev_all = s_model.predict(X_test)
  pp = p_freq * sev_all
  combo_name = f"{f_name} + {s_name}"
  result = evaluate_pure_premium(actual_loss, pp, model_name=combo_name)
  pp_results.append(result)
  if result["Normalized_Gini"] > best_gini:
   best_gini = result["Normalized_Gini"]
   best_combo_name = combo_name
   best_pp = pp

# Add all Tweedie single-model results
for tr in tweedie_results:
 pp_results.append(tr)

# Display full comparison
pp_df = compare_models(pp_results)
print("=== Pure Premium - Full Model Comparison ===\n")
print(pp_df.sort_values("Normalized_Gini", ascending=False).to_string())
print(f"\nBest two-part combo: {best_combo_name} (Gini={best_gini:.4f})")

=== Pure Premium - Full Model Comparison ===

                                   Total_Predicted     Total_Actual  Calibration_Ratio  Normalized_Gini
Model                                                                                                  
GLM (Logistic) + CatBoost         503,773,412.0000 523,096,888.0000             0.9631           0.6812
GLM (Logistic) + LightGBM (Gamma) 497,222,510.0000 523,096,888.0000             0.9505           0.6805
GLM (Logistic) + XGBoost (Gamma)  497,179,436.0000 523,096,888.0000             0.9505           0.6803
CatBoost + CatBoost               500,214,015.0000 523,096,888.0000             0.9563           0.6801
CatBoost + LightGBM (Gamma)       493,423,161.0000 523,096,888.0000             0.9433           0.6795
CatBoost + XGBoost (Gamma)        493,348,220.0000 523,096,888.0000             0.9431           0.6793
XGBoost + CatBoost                499,601,987.0000 523,096,888.0000             0.9551           0.6790
GLM (Logistic) + R

In [13]:
# Top-10 pure premium results sorted by Gini
print("=== Top 10 Model Combinations by Normalized Gini ===\n")
top10 = pp_df.sort_values("Normalized_Gini", ascending=False).head(10)
print(top10[["Calibration_Ratio", "Normalized_Gini"]].to_string())

=== Top 10 Model Combinations by Normalized Gini ===

                                   Calibration_Ratio  Normalized_Gini
Model                                                                
GLM (Logistic) + CatBoost                     0.9631           0.6812
GLM (Logistic) + LightGBM (Gamma)             0.9505           0.6805
GLM (Logistic) + XGBoost (Gamma)              0.9505           0.6803
CatBoost + CatBoost                           0.9563           0.6801
CatBoost + LightGBM (Gamma)                   0.9433           0.6795
CatBoost + XGBoost (Gamma)                    0.9431           0.6793
XGBoost + CatBoost                            0.9551           0.6790
GLM (Logistic) + Random Forest                0.9634           0.6788
LightGBM + CatBoost                           0.9552           0.6787
XGBoost + LightGBM (Gamma)                    0.9427           0.6782


---
## 6. Best Model Deep-Dive - Lift Chart & Feature Importance

In [14]:
# Lift chart for best two-part model (Plotly)
print(f"Best Two-Part Model: {best_combo_name}")
print(f"Normalized Gini: {best_gini:.4f}\n")

def plotly_lift_chart(actual, predicted, title):
 ev = pd.DataFrame({"pred": predicted, "actual": actual})
 ev["decile"] = pd.qcut(ev.pred.rank(method="first"), 10, labels=False) + 1
 by_dec = ev.groupby("decile").agg(
  mean_pred=("pred", "mean"), mean_actual=("actual", "mean")).reset_index()
 fig = go.Figure()
 fig.add_trace(go.Bar(x=by_dec.decile, y=by_dec.mean_actual, name="Actual Loss",
       marker_color="#1f6f8b", offsetgroup=0))
 fig.add_trace(go.Bar(x=by_dec.decile, y=by_dec.mean_pred, name="Predicted Premium",
       marker_color="#9bbcd1", offsetgroup=1))
 fig.update_layout(barmode="group", template="plotly_white", height=400,
      title=title, xaxis_title="Predicted-risk decile (10 = riskiest)",
      yaxis_title="Mean loss per policy (USD)")
 fig.show()
 lift = by_dec.mean_actual.iloc[-1] / max(by_dec.mean_actual.iloc[0], 1)
 print(f"Top vs bottom decile actual-loss lift: {lift:,.0f}x")

plotly_lift_chart(actual_loss, best_pp, f"Lift Chart - {best_combo_name}")

Best Two-Part Model: GLM (Logistic) + CatBoost
Normalized Gini: 0.6812



Top vs bottom decile actual-loss lift: 292x


# Lift chart for Tweedie (single model) for comparison
plotly_lift_chart(actual_loss, tweedie_pred, "Lift Chart - Tweedie GLM (Single Model)")

In [16]:
# Feature importance from the best frequency and severity models (Plotly)
best_freq_name = best_combo_name.split(" + ")[0]
best_sev_name = best_combo_name.split(" + ")[1]

def plotly_feature_importance(model, feature_names, title):
 if hasattr(model, "feature_importances_"):
  imp = pd.Series(model.feature_importances_, index=feature_names).sort_values()
 elif hasattr(model, "coef_"):
  coefs = np.abs(model.coef_[0])
  imp = pd.Series(coefs / coefs.sum(), index=feature_names).sort_values()
 else:
  print("No feature importance available"); return
 fig = px.bar(x=imp.values, y=imp.index, orientation="h",
     color_discrete_sequence=["#1f6f8b"], title=title)
 fig.update_layout(template="plotly_white", height=350,
      xaxis_title="Importance", yaxis_title="")
 fig.show()

print(f"Feature Importance - Frequency: {best_freq_name}")
plotly_feature_importance(freq_models[best_freq_name], FEATURES,
       f"Frequency Feature Importance ({best_freq_name})")

print(f"\nFeature Importance - Severity: {best_sev_name}")
plotly_feature_importance(sev_models[best_sev_name], FEATURES,
       f"Severity Feature Importance ({best_sev_name})")

Feature Importance - Frequency: GLM (Logistic)



Feature Importance - Severity: CatBoost


---
## 7. Pricing Framework

Model output = **technical price floor**. Actual premium adds:

$$\text{Premium} = \text{Pure premium} + \text{expense load} + \text{profit load} + \text{risk margin}$$

Important bits:
1. Use predicted pure premium to rank and segment policies
2. Calibration ratio ~1.0 means aggregate predictions match reality
3. Risk margin should be based on tail risk (VaR / high quantile), not the mean
4. Keep the GLM around for regulatory filings - it's auditable

---
## 8. Summary

In [17]:
# Final summary table
print("=" * 70)
print("FINAL MODEL COMPARISON - PURE PREMIUM")
print("=" * 70)

# Show top 5 + Tweedie
final_df = pp_df.sort_values("Normalized_Gini", ascending=False)
print("\n--- Top 5 Two-Part Combinations ---")
top5_twopart = final_df[final_df.index != "Tweedie GLM (Single)"].head(5)
print(top5_twopart[["Calibration_Ratio", "Normalized_Gini"]].to_string())

print("\n--- Tweedie Single Model ---")
tweedie_row = final_df[final_df.index == "Tweedie GLM (Single)"]
print(tweedie_row[["Calibration_Ratio", "Normalized_Gini"]].to_string())

print("\n" + "=" * 70)
print(f"BEST OVERALL: {final_df.index[0]}")
print(f" Calibration: {final_df.iloc[0]['Calibration_Ratio']:.4f}")
print(f" Norm. Gini: {final_df.iloc[0]['Normalized_Gini']:.4f}")
print("=" * 70)

FINAL MODEL COMPARISON - PURE PREMIUM

--- Top 5 Two-Part Combinations ---
                                   Calibration_Ratio  Normalized_Gini
Model                                                                
GLM (Logistic) + CatBoost                     0.9631           0.6812
GLM (Logistic) + LightGBM (Gamma)             0.9505           0.6805
GLM (Logistic) + XGBoost (Gamma)              0.9505           0.6803
CatBoost + CatBoost                           0.9563           0.6801
CatBoost + LightGBM (Gamma)                   0.9433           0.6795

--- Tweedie Single Model ---
Empty DataFrame
Columns: [Calibration_Ratio, Normalized_Gini]
Index: []

BEST OVERALL: GLM (Logistic) + CatBoost
 Calibration: 0.9631
 Norm. Gini: 0.6812


### Takeaways

| Approach | Pros | Cons |
|---|---|---|
| **GLM** | Interpretable, calibrated, easy to file | Misses non-linear interactions |
| **XGBoost** | Good on severity, picks up interactions | Slightly less calibrated freq |
| **LightGBM** | Fast, competitive | Similar to XGBoost here |
| **CatBoost** | Handles categoricals well | Slower grid search |
| **Random Forest** | Stable, low variance | Tends to oversmooth tails |
| **Tweedie (single)** | One model, handles zeros | Linear, limited ranking power |

Best approach: use the top two-part combination for production, keep GLM as the interpretable benchmark for filings and monitoring. Tweedie is a good sanity check.